In [1]:
import pandas as pd

diplome = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\diplome_region.csv")
chomage = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\chomage_format_long.csv")
creation_per_1000 = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\creation_per_1000.csv")
salaires = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\salaires.csv", sep=';')
population = pd.read_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\population_par_region_annee.csv")

chomage=chomage.rename(columns={'Region':'region_nom', 'TIME_VALUE':'TIME_PERIOD'})

salaires=salaires.rename(columns={'REGION_NOM':'region_nom', 'ANNEE1':'TIME_PERIOD'})


In [35]:

regression = pd.merge(creation_per_1000, diplome[['Pourcentage_diplomes_superieur', 'Pourcentage_aucun_diplome_ou_certificat_d_etudes_primaires','Pourcentage_Baccalauréat / Brevet professionnel ou équivalent', 'region_nom']], on='region_nom', how='left')

regression = pd.merge(regression, chomage[['Taux de chômage par région', 'region_nom', 'TIME_PERIOD']], on=['region_nom', 'TIME_PERIOD'], how='left') 

regression=pd.merge(regression, population[['variation_population_pourcentage', 'densité de population', 'region_nom', 'TIME_PERIOD']], on=['region_nom', 'TIME_PERIOD'], how='left')


#on va garder uniquement les colonnes utiles pour la régression
#fait le choix de garder 'Pourcentage_diplomes_superieur' pour la variable sur les diplômes
regression_finale=regression[['creations_per_1000', 'Pourcentage_diplomes_superieur', 'Taux de chômage par région', 'variation_population_pourcentage', 'densité de population']]

#envoyer regression_finale en csv
regression_finale.to_csv(r"C:/Users\thoma\OneDrive\Documents\ENSAE\2A\Python-pour-la-data-science\Projet\python_DS_2A\data\Variables explicatives\regression1.csv", index=False)


In [33]:
import statsmodels.api as sm
import pandas as pd

# 1. Définition des variables
# Y = La variable cible (à expliquer)
y = regression_finale['creations_per_1000']

# X = Les variables explicatives
features = [
    'Taux de chômage par région', 
    'Pourcentage_diplomes_superieur',     
    'variation_population_pourcentage', 
    'densité de population'
]
X = regression_finale[features]

# 2. Ajout de la constante (Intercept)
# C'est crucial : sans ça, la droite de régression est forcée de passer par 0
X = sm.add_constant(X)

# 3. Création et ajustement du modèle (MCO = Moindres Carrés Ordinaires)
model = sm.OLS(y, X)
results = model.fit()

# 4. Affichage du rapport complet
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:     creations_per_1000   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.563
Method:                 Least Squares   F-statistic:                     42.58
Date:              lun., 15 déc. 2025   Prob (F-statistic):           1.70e-22
Time:                        21:57:23   Log-Likelihood:                -316.14
No. Observations:                 130   AIC:                             642.3
Df Residuals:                     125   BIC:                             656.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

Idée ça peut être intéressant de prendre en compte l'âge d'une région en créant une variable binaire par exemple.

In [ ]:
salaires=pd